# 03 — Training protocol and safety

**Tujuan.** Audit konfigurasi sebelum GPU training, tanpa menjalankannya. Paper §IV-B hlm. 91509 menyatakan split 8:1:1, input 512, LR 0,000186, batch 16, 80 epoch. Seed 42, AdamW, CE, AMP, clipping, aturan checkpoint, dan strategi oversampling adalah `ASSUMED`. Batch efektif 16 dengan microbatch lebih kecil mengubah statistik BatchNorm.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.experiments import Protocol, CAMPAIGNS, load_protocol, matrix
print("Paper explicit: input=512, LR=0.000186, batch=16, epochs=80, split=8:1:1")
print("Runner defaults (assumptions marked in docs):", Protocol())
if ARTIFACT_ROOT:
    for campaign in CAMPAIGNS:
        try:
            protocol, fingerprint = load_protocol(ARTIFACT_ROOT, campaign)
            print(campaign, "AVAILABLE", fingerprint, "balancing", protocol.balancing)
        except (FileNotFoundError, ValueError) as exc:
            print(campaign, "PENDING:", exc)
else:
    print("PENDING: no local campaign artifact root; no training is launched")

Paper explicit: input=512, LR=0.000186, batch=16, epochs=80, split=8:1:1
Runner defaults (assumptions marked in docs): Protocol(seed=42, epochs=80, learning_rate=0.000186, effective_batch_size=16, micro_batch_size=16, accumulation=1, preprocessing='paper_described_historical_v1', augment_train=True, balancing='none', optimizer='AdamW', weight_decay=0.0, loss='CrossEntropyLoss', amp=True, grad_clip_norm=1.0, checkpoint_metric='validation_macro_f1', checkpoint_tiebreak='validation_loss', num_workers=0)
PENDING: no local campaign artifact root; no training is launched


**Metode dan batasan.** `paper_v1` di KC sudah dibuat dan tidak boleh ditimpa. `balanced_v1` harus memiliki direktori, protokol, fingerprint, dan checkpoint terpisah. Runner melakukan preflight manifest tanpa membuka gambar test, memilih checkpoint dengan validation macro-F1, dan hanya mengakses test lewat perintah final-test eksplisit. Pilot satu epoch memakai `--max-epochs-per-invocation 1`; perintah lengkap ada di rencana eksperimen.